# 05 — Final Model Training & Ensemble

This notebook makes the **final modeling decision** for the Moneyball season-wins project.

By this point:

- `02_feature_engineering.ipynb` created the canonical model-ready features;
- `03_model_feature_selection.ipynb` selected each model's best feature set;
- `04_cluster_feature_experiment.ipynb` showed that KMeans features do not add enough value to keep in the main pipeline.

The remaining question is:

> **Should the final prediction use one strong standalone model, or can a linear + nonlinear ensemble reduce MAE further?**

### Objectives

1. Reuse each candidate model's **best feature set from 03**.
2. Generate leakage-safe **out-of-fold (OOF) predictions** for ElasticNet, Ridge, HGB, and LightGBM.
3. Compare standalone MAE and **residual correlation** to see whether the models make different mistakes.
4. Test a small set of weighted ensembles, especially **linear + nonlinear** combinations.
5. Run a **secondary chronological holdout** as a sanity check.
6. Select the final standalone model or ensemble, fit it on all training rows, and create the submission.

### Important decision

LightGBM is kept only as an **ensemble challenger**. It was weaker as a standalone model in 03, but a weaker model can still help if its residuals are sufficiently different.

### Outputs

Written to `outputs/final_model/`:

- `05_oof_predictions.csv`
- `05_model_results.csv`
- `05_residual_correlations.csv`
- `05_chronological_results.csv`
- `05_ensemble_results.csv`
- `05_final_decision.csv`

Final submission:

- `outputs/submissions/submission_predict.csv`


## 1. Setup and load inputs

05 does not rebuild features and does not redo feature selection. It consumes the decisions made upstream.


In [1]:
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GroupKFold

from moneyball import project_config as cfg
from moneyball import feature_sets as fs
from moneyball import model_factory as mf

cfg.configure_notebook()
cfg.ensure_project_dirs()

TRAIN_FE_PATH        = cfg.TRAIN_FE_PATH
PRED_FE_PATH         = cfg.PRED_FE_PATH
RESULTS_03_PATH      = cfg.MODEL_RESULT_PATH
CLUSTER_SUMMARY_PATH = cfg.CLUSTER_SUMMARY_PATH

OOF_PATH             = cfg.OOF_PATH
MODEL_RESULT_PATH    = cfg.FINAL_RESULT_PATH
RESIDUAL_CORR_PATH   = cfg.RESIDUAL_CORR_PATH
CHRONO_PATH          = cfg.CHRONO_PATH
ENSEMBLE_PATH        = cfg.ENSEMBLE_PATH
DECISION_PATH        = cfg.DECISION_PATH
SUBMISSION_PATH      = cfg.SUBMISSION_PATH

print("Train FE        :", TRAIN_FE_PATH)
print("Prediction FE   :", PRED_FE_PATH)
print("03 results      :", RESULTS_03_PATH)
print("Final Submission:", SUBMISSION_PATH)

Train FE        : /home/shpang/devs/ntu/projects/baseball_v2/data/processed/train_fe.csv
Prediction FE   : /home/shpang/devs/ntu/projects/baseball_v2/data/processed/pred_fe.csv
03 results      : /home/shpang/devs/ntu/projects/baseball_v2/outputs/model_selection/03_model_feature_results.csv
Final Submission: /home/shpang/devs/ntu/projects/baseball_v2/outputs/submissions/submission.csv


In [2]:
train_fe = pd.read_csv(TRAIN_FE_PATH)
pred_fe = pd.read_csv(PRED_FE_PATH)

if not RESULTS_03_PATH.exists():
    raise FileNotFoundError(
        "03_model_feature_results.csv was not found. Run 03 first."
    )

results_03 = pd.read_csv(RESULTS_03_PATH)

FEATURE_SETS = fs.get_candidate_feature_sets(train_fe)
fs.validate_feature_sets(train_fe, pred_fe, FEATURE_SETS)

print("Train shape:", train_fe.shape)
print("Pred shape :", pred_fe.shape)

if CLUSTER_SUMMARY_PATH.exists():
    print("\n04 cluster summary (reference only):")
    display(pd.read_csv(CLUSTER_SUMMARY_PATH))

Train shape: (1812, 78)
Pred shape : (453, 76)

04 cluster summary (reference only):


,model,feature_set,variant,n_clusters,baseline_mae,cluster_mae,delta_mae,cluster_mae_std,folds_improved,best_variant,best_k,best_mae,best_mae_std,total_n_features
0,elasticnet,all_domain_raw,cluster_k8,8.0,2.727792,2.734371,0.006579,0.092397,2,baseline,8.0,2.727792,0.089080,72
1,hgb,all_domain_raw,cluster_k4,4.0,3.070202,3.064434,-0.005768,0.059647,4,cluster_k4,4.0,3.064434,0.059647,76
2,ridge,all_domain_raw,cluster_k8,8.0,2.729300,2.737869,0.008569,0.085010,1,baseline,8.0,2.729300,0.101873,72


## 2. Reconstruct each candidate's best configuration from 03

We carry four models into the final analysis:

- **ElasticNet** — strongest linear result;
- **Ridge** — essentially tied with ElasticNet;
- **HGB** — strongest nonlinear model;
- **LightGBM** — weaker standalone, kept only to test ensemble diversity.

Each model keeps the feature set on which it actually performed best in 03.


In [3]:
CANDIDATE_MODELS = ["elasticnet", "ridge", "hgb", "lightgbm"]

missing = sorted(set(CANDIDATE_MODELS) - set(results_03["model"]))
if missing:
    raise ValueError(f"03 results are missing candidate models: {missing}")

best_config = (
    results_03[results_03["model"].isin(CANDIDATE_MODELS)]
    .sort_values(["mae_mean", "mae_std"])
    .groupby("model", as_index=False)
    .first()
    .set_index("model")
    .loc[CANDIDATE_MODELS]
    .reset_index()
)

unknown_sets = sorted(set(best_config["feature_set"]) - set(FEATURE_SETS))
if unknown_sets:
    raise ValueError(f"Unknown feature sets from 03: {unknown_sets}")

display(best_config[
    ["model", "feature_set", "n_features", "mae_mean", "mae_std"]
].round(4))

,model,feature_set,n_features,mae_mean,mae_std
0,elasticnet,all_domain_raw,72,2.7278,0.0891
1,ridge,all_domain_raw,72,2.7293,0.1019
2,hgb,all_domain_raw,72,3.0702,0.0657
3,lightgbm,core_pitch,18,3.1159,0.0517


## 3. Primary validation — OOF predictions

We keep the same primary validation design used upstream:

- 5-fold `GroupKFold`
- grouped by `meta_yearID`
- target = `target_W`
- metric = MAE in wins

The difference now is that we save **one prediction for every training row**. Those OOF predictions let us compare residuals and test ensembles fairly.


In [4]:
TARGET_COL = "target_W"
GROUP_COL = "meta_yearID"
N_SPLITS = 5

y = train_fe[TARGET_COL].astype(float)
groups = train_fe[GROUP_COL]
cv = GroupKFold(n_splits=N_SPLITS)

fold_assignment = np.zeros(len(train_fe), dtype=int)
for fold_no, (_, valid_idx) in enumerate(cv.split(train_fe, y, groups=groups), start=1):
    fold_assignment[valid_idx] = fold_no

assert (fold_assignment > 0).all()

In [5]:
def generate_oof(model_name, feature_set_name):
    """Generate one leakage-safe OOF prediction for every training row."""
    features = FEATURE_SETS[feature_set_name]
    X = train_fe[features]
    pred = np.full(len(train_fe), np.nan)

    with warnings.catch_warnings():
        warnings.simplefilter("once", ConvergenceWarning)

        for train_idx, valid_idx in cv.split(X, y, groups=groups):
            model = mf.MODEL_FACTORIES[model_name]()
            model.fit(X.iloc[train_idx], y.iloc[train_idx])
            pred[valid_idx] = model.predict(X.iloc[valid_idx])

    if np.isnan(pred).any():
        raise RuntimeError(f"Incomplete OOF predictions for {model_name}")

    return pred


oof = pd.DataFrame({
    "meta_ID": train_fe["meta_ID"],
    "meta_yearID": train_fe["meta_yearID"],
    "fold": fold_assignment,
    "actual_W": y,
})

for _, row in best_config.iterrows():
    model_name = row["model"]
    print(f"OOF: {model_name} + {row['feature_set']}")
    oof[f"pred_{model_name}"] = generate_oof(
        model_name,
        row["feature_set"],
    )

oof.to_csv(OOF_PATH, index=False)
display(oof.head())

OOF: elasticnet + all_domain_raw
OOF: ridge + all_domain_raw
OOF: hgb + all_domain_raw
OOF: lightgbm + core_pitch


/home/shpang/miniconda3/envs/mball/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/shpang/miniconda3/envs/mball/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/shpang/miniconda3/envs/mball/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/shpang/miniconda3/envs/mball/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/shpang/miniconda3/envs/mball/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, 

,meta_ID,meta_yearID,fold,actual_W,pred_elasticnet,pred_ridge,pred_hgb,pred_lightgbm
0,317,1935,4,78.0,77.143541,77.368252,76.092919,75.747229
1,2162,1993,5,86.0,89.314393,89.810519,90.534685,90.244798
2,1895,2016,5,86.0,88.652666,88.678974,90.251393,90.156696
3,428,1938,5,89.0,89.406069,89.703694,88.434008,90.764088
4,375,1996,3,85.0,81.157340,81.264319,83.778098,82.843575


## 4. Standalone results and residual correlation

Residuals are:

`actual wins - OOF prediction`

A weaker nonlinear model can still be useful in an ensemble if it makes **different mistakes** from the strongest linear model.


In [6]:
model_rows = []
residuals = pd.DataFrame(index=oof.index)

for _, row in best_config.iterrows():
    model_name = row["model"]
    pred_col = f"pred_{model_name}"

    fold_mae = []
    for fold_no in range(1, N_SPLITS + 1):
        f = oof[oof["fold"] == fold_no]
        fold_mae.append(mean_absolute_error(f["actual_W"], f[pred_col]))

    model_rows.append({
        "model": model_name,
        "feature_set": row["feature_set"],
        "n_features": int(row["n_features"]),
        "oof_mae": mean_absolute_error(oof["actual_W"], oof[pred_col]),
        "fold_mae_mean": float(np.mean(fold_mae)),
        "fold_mae_std": float(np.std(fold_mae)),
    })

    residuals[model_name] = oof["actual_W"] - oof[pred_col]

model_results = pd.DataFrame(model_rows).sort_values("oof_mae").reset_index(drop=True)
residual_corr = residuals.corr()

model_results.to_csv(MODEL_RESULT_PATH, index=False)
residual_corr.to_csv(RESIDUAL_CORR_PATH)

print("Standalone OOF results")
display(model_results.round(4))

print("Residual correlation")
display(residual_corr.round(3))

Standalone OOF results


,model,feature_set,n_features,oof_mae,fold_mae_mean,fold_mae_std
0,elasticnet,all_domain_raw,72,2.7275,2.7278,0.0891
1,ridge,all_domain_raw,72,2.7290,2.7293,0.1019
2,hgb,all_domain_raw,72,3.0703,3.0702,0.0657
3,lightgbm,core_pitch,18,3.1160,3.1159,0.0517


Residual correlation


,elasticnet,ridge,hgb,lightgbm
elasticnet,1.000,0.994,0.873,0.866
ridge,0.994,1.000,0.854,0.846
hgb,0.873,0.854,1.000,0.936
lightgbm,0.866,0.846,0.936,1.000


### How to interpret residual correlation

- Ridge and ElasticNet are expected to be highly correlated because both are regularized linear models.
- HGB and LightGBM only become useful ensemble partners if their residuals are sufficiently different.
- Correlation alone is not enough: the blend must still improve MAE.


## 5. Controlled ensemble test

The better of Ridge and ElasticNet becomes the **linear anchor**.

We test each other model as a small-weight partner at:

5%, 10%, 15%, 20%, 25%, and 30%.

The nonlinear models are materially weaker standalone, so if they help, they should usually act as a **small diversity correction**, not dominate the prediction.


In [15]:
linear_anchor = (
    model_results[model_results["model"].isin(["elasticnet", "ridge"])]
    .sort_values("oof_mae")
    .iloc[0]["model"]
)

partners = [m for m in CANDIDATE_MODELS if m != linear_anchor]
# PARTNER_WEIGHTS = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30]
PARTNER_WEIGHTS = [
    0.05, 0.10, 0.15, 0.20, 0.25, 0.30,
    0.35, 0.40, 0.45, 0.50, 0.55, 0.60,
    0.65, 0.70, 0.75, 0.80, 0.85, 0.90,
]

actual = oof["actual_W"].to_numpy()
anchor_pred = oof[f"pred_{linear_anchor}"].to_numpy()

ensemble_rows = [{
    "configuration": linear_anchor,
    "anchor_model": linear_anchor,
    "partner_model": None,
    "anchor_weight": 1.0,
    "partner_weight": 0.0,
    "oof_mae": mean_absolute_error(actual, anchor_pred),
}]

for partner in partners:
    partner_pred = oof[f"pred_{partner}"].to_numpy()

    for pw in PARTNER_WEIGHTS:
        aw = 1.0 - pw
        blended = aw * anchor_pred + pw * partner_pred

        ensemble_rows.append({
            "configuration": f"{linear_anchor}_{aw:.2f}__{partner}_{pw:.2f}",
            "anchor_model": linear_anchor,
            "partner_model": partner,
            "anchor_weight": aw,
            "partner_weight": pw,
            "oof_mae": mean_absolute_error(actual, blended),
        })

ensemble_results = (
    pd.DataFrame(ensemble_rows)
    .sort_values("oof_mae")
    .reset_index(drop=True)
)

print("Linear anchor:", linear_anchor)
display(ensemble_results.head(15).round(4))

Linear anchor: elasticnet


,configuration,anchor_model,partner_model,anchor_weight,partner_weight,oof_mae
0,elasticnet_0.55__ridge_0.45,elasticnet,ridge,0.55,0.45,2.7244
1,elasticnet_0.50__ridge_0.50,elasticnet,ridge,0.50,0.50,2.7244
2,elasticnet_0.60__ridge_0.40,elasticnet,ridge,0.60,0.40,2.7244
3,elasticnet_0.45__ridge_0.55,elasticnet,ridge,0.45,0.55,2.7245
4,elasticnet_0.65__ridge_0.35,elasticnet,ridge,0.65,0.35,2.7245
5,elasticnet_0.40__ridge_0.60,elasticnet,ridge,0.40,0.60,2.7247
6,elasticnet_0.70__ridge_0.30,elasticnet,ridge,0.70,0.30,2.7247
7,elasticnet_0.75__ridge_0.25,elasticnet,ridge,0.75,0.25,2.7249
8,elasticnet_0.35__ridge_0.65,elasticnet,ridge,0.35,0.65,2.7250
9,elasticnet_0.80__ridge_0.20,elasticnet,ridge,0.80,0.20,2.7253


## 6. Secondary chronological holdout

GroupKFold remains the primary comparison because it is consistent with 03 and 04.

As a final sanity check, we also train on the earliest 80% of unique seasons and validate on the latest 20%.

This asks a different question:

> Does the selected model still behave sensibly when predicting later historical seasons?


In [16]:
years = np.array(sorted(train_fe[GROUP_COL].unique()))
split_at = int(len(years) * 0.80)

train_years = years[:split_at]
valid_years = years[split_at:]

train_mask = train_fe[GROUP_COL].isin(train_years)
valid_mask = train_fe[GROUP_COL].isin(valid_years)

chrono_actual = y.loc[valid_mask].to_numpy()
chrono_pred = {}

config_lookup = best_config.set_index("model").to_dict("index")

for model_name in CANDIDATE_MODELS:
    feature_set_name = config_lookup[model_name]["feature_set"]
    features = FEATURE_SETS[feature_set_name]

    model = mf.MODEL_FACTORIES[model_name]()
    model.fit(train_fe.loc[train_mask, features], y.loc[train_mask])
    chrono_pred[model_name] = model.predict(train_fe.loc[valid_mask, features])

print(
    f"Chronological train: {train_years.min()}–{train_years.max()} "
    f"({train_mask.sum()} rows)"
)
print(
    f"Chronological valid: {valid_years.min()}–{valid_years.max()} "
    f"({valid_mask.sum()} rows)"
)

Chronological train: 1904–1992 (1275 rows)
Chronological valid: 1993–2016 (537 rows)


/home/shpang/miniconda3/envs/mball/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [17]:
chrono_rows = []

for _, row in ensemble_results.iterrows():
    anchor = row["anchor_model"]
    partner = row["partner_model"]

    prediction = row["anchor_weight"] * chrono_pred[anchor]

    if not pd.isna(partner):
        prediction = prediction + row["partner_weight"] * chrono_pred[partner]

    chrono_rows.append({
        "configuration": row["configuration"],
        "chrono_mae": mean_absolute_error(chrono_actual, prediction),
    })

chrono_results = pd.DataFrame(chrono_rows)

ensemble_results = ensemble_results.merge(
    chrono_results,
    on="configuration",
    how="left",
)

ensemble_results.to_csv(ENSEMBLE_PATH, index=False)
chrono_results.to_csv(CHRONO_PATH, index=False)

display(
    ensemble_results[
        ["configuration", "oof_mae", "chrono_mae",
         "anchor_model", "partner_model", "partner_weight"]
    ].head(15).round(4)
)

,configuration,oof_mae,chrono_mae,anchor_model,partner_model,partner_weight
0,elasticnet_0.55__ridge_0.45,2.7244,2.3687,elasticnet,ridge,0.45
1,elasticnet_0.50__ridge_0.50,2.7244,2.3678,elasticnet,ridge,0.50
2,elasticnet_0.60__ridge_0.40,2.7244,2.3697,elasticnet,ridge,0.40
3,elasticnet_0.45__ridge_0.55,2.7245,2.3668,elasticnet,ridge,0.55
4,elasticnet_0.65__ridge_0.35,2.7245,2.3707,elasticnet,ridge,0.35
5,elasticnet_0.40__ridge_0.60,2.7247,2.3659,elasticnet,ridge,0.60
6,elasticnet_0.70__ridge_0.30,2.7247,2.3718,elasticnet,ridge,0.30
7,elasticnet_0.75__ridge_0.25,2.7249,2.3729,elasticnet,ridge,0.25
8,elasticnet_0.35__ridge_0.65,2.7250,2.3651,elasticnet,ridge,0.65
9,elasticnet_0.80__ridge_0.20,2.7253,2.3741,elasticnet,ridge,0.20


## 7. Final decision

The **primary ranking metric is OOF MAE**.

The chronological holdout is a secondary check.

### Decision rule

- Prefer the OOF winner when the chronological result is also sensible.
- If two configurations are effectively tied, prefer the simpler one.
- Do not keep a nonlinear model just because it is different; it must improve the blend.
- Do not overreact to tiny differences that are small relative to normal fold variation.

The next cell proposes the OOF winner automatically, but keeps the final choice explicit so the decision remains human-readable.


In [18]:
AUTO_BEST_CONFIG = ensemble_results.iloc[0]["configuration"]

# Human-in-the-loop final choice.
# Change this only if the chronological result or a simplicity argument
# gives a clear reason to override the OOF winner.
FINAL_CONFIG = AUTO_BEST_CONFIG

selected = ensemble_results[
    ensemble_results["configuration"] == FINAL_CONFIG
].iloc[0]

print("Automatic OOF winner :", AUTO_BEST_CONFIG)
print("Selected final config:", FINAL_CONFIG)

display(selected[
    ["configuration", "oof_mae", "chrono_mae",
     "anchor_model", "partner_model",
     "anchor_weight", "partner_weight"]
].to_frame("value"))

Automatic OOF winner : elasticnet_0.55__ridge_0.45
Selected final config: elasticnet_0.55__ridge_0.45


,value
configuration,elasticnet_0.55__ridge_0.45
oof_mae,2.724362
chrono_mae,2.368721
anchor_model,elasticnet
partner_model,ridge
anchor_weight,0.55
partner_weight,0.45


## 8. Float vs rounded predictions

Wins are integers in the historical data, but MAE can still be lower with continuous predictions.

We therefore compare the selected configuration in **both validation views**:

- OOF MAE using continuous predictions;
- OOF MAE after rounding to the nearest win;
- chronological MAE using continuous predictions;
- chronological MAE after rounding to the nearest win.

### Decision rule

Use rounded predictions only when rounding improves the primary OOF result **and does not make the chronological sanity check worse**.

This avoids choosing integer predictions based on one validation view alone.

In [19]:
anchor = selected["anchor_model"]
partner = selected["partner_model"]

# -------------------------------------------------------------
# OOF prediction for the selected configuration
# -------------------------------------------------------------
selected_oof = (
    selected["anchor_weight"]
    * oof[f"pred_{anchor}"].to_numpy()
)

if not pd.isna(partner):
    selected_oof += (
        selected["partner_weight"]
        * oof[f"pred_{partner}"].to_numpy()
    )

float_oof_mae = mean_absolute_error(
    actual,
    selected_oof,
)

rounded_oof_mae = mean_absolute_error(
    actual,
    np.rint(selected_oof),
)

# -------------------------------------------------------------
# Chronological prediction for the same selected configuration
# -------------------------------------------------------------
selected_chrono = (
    selected["anchor_weight"]
    * chrono_pred[anchor]
)

if not pd.isna(partner):
    selected_chrono += (
        selected["partner_weight"]
        * chrono_pred[partner]
    )

float_chrono_mae = mean_absolute_error(
    chrono_actual,
    selected_chrono,
)

rounded_chrono_mae = mean_absolute_error(
    chrono_actual,
    np.rint(selected_chrono),
)

# Primary rule: rounding must improve OOF MAE.
# Secondary rule: it must not make the chronological check worse.
USE_ROUNDED_PREDICTIONS = (
    rounded_oof_mae < float_oof_mae
    and rounded_chrono_mae <= float_chrono_mae
)

print(f"Float OOF MAE       : {float_oof_mae:.4f}")
print(f"Rounded OOF MAE     : {rounded_oof_mae:.4f}")
print(f"Float Chrono MAE    : {float_chrono_mae:.4f}")
print(f"Rounded Chrono MAE  : {rounded_chrono_mae:.4f}")
print(
    "Submission form    :",
    "rounded wins"
    if USE_ROUNDED_PREDICTIONS
    else "continuous wins",
)

Float OOF MAE       : 2.7244
Rounded OOF MAE     : 2.7081
Float Chrono MAE    : 2.3687
Rounded Chrono MAE  : 2.3557
Submission form    : rounded wins


## 9. Fit the selected configuration on all training rows

Only the model(s) with non-zero final weight are fitted.

Each model keeps its own 03-selected feature set. This matters especially if LightGBM survives as an ensemble partner.


In [22]:
def fit_full_predict(model_name):
    config = config_lookup[model_name]
    features = FEATURE_SETS[config["feature_set"]]

    model = mf.MODEL_FACTORIES[model_name]()
    model.fit(train_fe[features], y)

    return model, model.predict(pred_fe[features])


final_models = {}
final_prediction = np.zeros(len(pred_fe), dtype=float)

anchor_model, anchor_pred = fit_full_predict(selected["anchor_model"])
final_models[selected["anchor_model"]] = anchor_model
final_prediction += selected["anchor_weight"] * anchor_pred

if not pd.isna(selected["partner_model"]):
    partner_model, partner_pred = fit_full_predict(selected["partner_model"])
    final_models[selected["partner_model"]] = partner_model
    final_prediction += selected["partner_weight"] * partner_pred

# Season wins cannot be below 0 or above games played.
final_prediction = np.clip(
    final_prediction,
    0,
    pred_fe["season_G"].to_numpy(),
)

if USE_ROUNDED_PREDICTIONS:
    final_prediction = np.rint(final_prediction)

print("Fitted models:", list(final_models))
print(
    "Prediction range:",
    float(final_prediction.min()),
    "to",
    float(final_prediction.max()),
)

Fitted models: ['elasticnet', 'ridge']
Prediction range: 44.0 to 109.0


## 10. Create the submission and save the decision record

The decision record captures the exact models, feature sets, weights, and prediction form used so the final submission can be reproduced later.


In [23]:
submission = pd.DataFrame({
    cfg.ID_COL: pred_fe["meta_ID"],
    cfg.TARGET_COL: final_prediction,
})

submission.to_csv(SUBMISSION_PATH, index=False)

decision = pd.DataFrame([{
    "final_configuration": FINAL_CONFIG,
    "oof_mae": selected["oof_mae"],
    "chronological_mae": selected["chrono_mae"],
    "anchor_model": selected["anchor_model"],
    "anchor_feature_set": config_lookup[selected["anchor_model"]]["feature_set"],
    "anchor_weight": selected["anchor_weight"],
    "partner_model": selected["partner_model"],
    "partner_feature_set": (
        None
        if pd.isna(selected["partner_model"])
        else config_lookup[selected["partner_model"]]["feature_set"]
    ),
    "partner_weight": selected["partner_weight"],
    "float_oof_mae": float_oof_mae,
    "rounded_oof_mae": rounded_oof_mae,
    "float_chronological_mae": float_chrono_mae,
    "rounded_chronological_mae": rounded_chrono_mae,
    "rounded_predictions": USE_ROUNDED_PREDICTIONS,
}])

decision.to_csv(DECISION_PATH, index=False)

print("Saved submission     :", SUBMISSION_PATH)
print("Saved decision record:", DECISION_PATH)

display(submission.head())
display(decision.T)

Saved submission     : /home/shpang/devs/ntu/projects/baseball_v2/outputs/submissions/submission.csv
Saved decision record: /home/shpang/devs/ntu/projects/baseball_v2/outputs/final_model/05_final_decision.csv


,ID,W
0,1756,70.0
1,1282,75.0
2,351,84.0
3,421,87.0
4,57,93.0


,0
final_configuration,elasticnet_0.55__ridge_0.45
oof_mae,2.724362
chronological_mae,2.368721
anchor_model,elasticnet
anchor_feature_set,all_domain_raw
anchor_weight,0.55
partner_model,ridge
partner_feature_set,all_domain_raw
partner_weight,0.45
float_oof_mae,2.724362


## 11. Final interpretation

After running the notebook, record the actual conclusion here in plain language.

### Observation

- Which standalone model had the best OOF MAE?
- How correlated were Ridge and ElasticNet residuals?
- Were HGB or LightGBM errors meaningfully different?
- Did a linear + nonlinear blend improve MAE?
- Did the winning configuration remain sensible on the chronological holdout?
- Did rounding help or hurt?

### Decision

State the exact final standalone model or ensemble and its weights.

### Reasoning

Keep the reasoning tied to validation evidence.

If an ensemble wins, the reason should not simply be “ensembles are better.” It should show that the linear anchor was strong and the partner contributed sufficiently different errors to reduce OOF MAE.

If no ensemble improves on the anchor, the correct decision is the simpler standalone model.
